# CAPI Ablation Study — Statistical Analysis

This for the five response-generation configurations
(LLM-only, LLM+RAG, LLM+RAG+PA at Top-1/3/5):

- **Inter-rater reliability** — quadratic-weighted Cohen's κ (reported separately; *not* part of any ranking).
- **RQ1** — how configuration affects **relevance** (BERTScore, ROUGE-L) and **correctness** (human, 3-point): descriptive statistics + effect sizes, and Friedman omnibus + Holm-corrected pairwise Wilcoxon signed-rank tests.
- **RQ2** — how well the automatic metrics (BERTScore, ROUGE-L) agree with human judgment, via per-response rank correlation (Spearman, Kendall).

**Measurement decisions**:
Incorrect=0 / Partially Correct=1 / Correct=2; the two raters are combined by *averaging* their ordinal codes.

Requires: `pandas`, `scipy`, `scikit-learn` (`pip install pandas scipy scikit-learn`).

In [ ]:
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon, spearmanr, kendalltau
from sklearn.metrics import cohen_kappa_score

In [ ]:
from pathlib import Path
import pandas as pd

# Repo root, resolved relative to this file — works regardless of where you run it from.
ROOT = Path(__file__).resolve().parents[1]   # notebook: Path.cwd().parents[0]
DATA = ROOT / "03_Data"

RATER1_CSV = pd.read_csv(DATA / "rater2_ratings.csv")
RATER2_CSV = pd.read_csv(DATA / "rater1_ratings.csv")
BERT_CSV = pd.read_csv(DATA / "Groundtruth_BERTScore_per_query.csv")
ROUGE_CSV = pd.read_csv(DATA / "Groundtruth_ROUGEL_per_query.csv")

RUN_TESTS = True   # set False for descriptive statistics only (no Friedman/Wilcoxon)

# canonical config -> exact column names in YOUR files
CONFIG_META = {
    "LLM-only": dict(bert="LLMOnly_BERTScore", rate="LLMOnly",  rouge="LLMOnly_ROUGEL"),
    "LLM+RAG":  dict(bert="RAG_BERTScore",     rate="LLM+RAG",  rouge="RAG_ROUGEL"),
    "Top-1":    dict(bert="Top1_BERTScore",    rate="Top1",     rouge="Top1_ROUGEL"),
    "Top-3":    dict(bert="Top3_BERTScore",    rate="Top3",     rouge="Top3_ROUGEL"),
    "Top-5":    dict(bert="Top5_BERTScore",    rate="Top5",     rouge="Top5_ROUGEL"),
}
CONFIG_ORDER = ["LLM-only", "LLM+RAG", "Top-1", "Top-3", "Top-5"]

In [ ]:
def normalize_id(df, who):
    """Accept 'Query' or 'QueryNumber' as the id column; unify to 'QueryNumber'."""
    for cand in ["QueryNumber", "Query", "query", "querynumber", "Query Number"]:
        if cand in df.columns:
            return df.rename(columns={cand: "QueryNumber"})
    raise ValueError(f"[{who}] no id column (looked for QueryNumber/Query). "
                     f"Have: {list(df.columns)}")

def to_code(v):
    if pd.isna(v):
        return np.nan
    s = " ".join(str(v).strip().lower().split())
    return {"incorrect": 0, "partially correct": 1, "correct": 2}.get(s, np.nan)

def holm(pvals):
    """Holm-Bonferroni step-down correction."""
    p = np.asarray(pvals, float); m = len(p)
    order = np.argsort(p); adj = np.empty(m); running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adj[idx] = min(running, 1.0)
    return adj

def kendalls_w(mat):
    n, k = mat.shape
    ranks = np.apply_along_axis(lambda r: pd.Series(r).rank().values, 1, mat)
    Rj = ranks.sum(axis=0); S = ((Rj - Rj.mean()) ** 2).sum()
    return 12 * S / (n ** 2 * (k ** 3 - k))

def rank_biserial(x, y):
    d = np.asarray(y, float) - np.asarray(x, float); d = d[d != 0]
    if d.size == 0:
        return 0.0
    ranks = pd.Series(np.abs(d)).rank().values
    rp, rm = ranks[d > 0].sum(), ranks[d < 0].sum(); tot = rp + rm
    return (rp - rm) / tot if tot else 0.0

def interpret_kappa(k):
    for thr, label in [(0.81,"almost perfect"),(0.61,"substantial"),
                       (0.41,"moderate"),(0.21,"fair"),(0.0,"slight")]:
        if k >= thr:
            return label
    return "poor"

In [ ]:
main_df = normalize_id(pd.read_csv(BERT_CSV), "bert")
r1 = normalize_id(pd.read_csv(RATER1_CSV), "rater1")
r2 = normalize_id(pd.read_csv(RATER2_CSV), "rater2")
rouge_df = normalize_id(pd.read_csv(ROUGE_CSV), "rouge") if ROUGE_CSV else main_df

frames, rouge_ok = {}, True
for cfg in CONFIG_ORDER:
    m = CONFIG_META[cfg]
    b  = main_df[["QueryNumber", m["bert"]]].rename(columns={m["bert"]: "bert"})
    a1 = r1[["QueryNumber", m["rate"]]].rename(columns={m["rate"]: "r1"})
    a2 = r2[["QueryNumber", m["rate"]]].rename(columns={m["rate"]: "r2"})
    d = b.merge(a1, on="QueryNumber").merge(a2, on="QueryNumber")
    if m["rouge"] in rouge_df.columns:
        d = d.merge(rouge_df[["QueryNumber", m["rouge"]]].rename(columns={m["rouge"]:"rouge"}),
                    on="QueryNumber", how="left")
        d["rouge"] = pd.to_numeric(d["rouge"], errors="coerce")
    else:
        rouge_ok = False
    d["bert"] = pd.to_numeric(d["bert"], errors="coerce")
    d["r1"] = d["r1"].map(to_code); d["r2"] = d["r2"].map(to_code)
    d["human"] = d[["r1", "r2"]].mean(axis=1)
    frames[cfg] = d

print("Loaded. n per configuration:", {c: len(frames[c]) for c in CONFIG_ORDER})
print("ROUGE-L available:", rouge_ok)
if not rouge_ok:
    print(">> ROUGE-L not found for all configs; ROUGE-L sections will be skipped.")

## Inter-rater reliability
Reported separately as measurement reliability; **not** part of any ranking.

In [ ]:
print(f"{'config':10s}  kappa (quadratic-weighted)")
for cfg in CONFIG_ORDER:
    d = frames[cfg].dropna(subset=["r1", "r2"])
    k = cohen_kappa_score(d["r1"].astype(int), d["r2"].astype(int), weights="quadratic")
    print(f"  {cfg:10s}  {k:6.3f}  ({interpret_kappa(k)}, n={len(d)})")

### Shared analysis functions (RQ1 and RQ2)

In [ ]:
def rq1_analysis(metric, label, do_tests=True):

    print("Descriptive statistics:")
    for cfg in CONFIG_ORDER:
        s = frames[cfg][metric].dropna()
        print(f"  {cfg:10s} mean={s.mean():8.4f}  sd={s.std(ddof=1):7.4f}  "
              f"median={s.median():7.4f}  n={len(s)}")

    if metric == "human":
        print("\nCorrectness distribution (both raters pooled):")
        for cfg in CONFIG_ORDER:
            codes = pd.concat([frames[cfg]["r1"], frames[cfg]["r2"]]).dropna()
            print(f"  {cfg:10s} Incorrect={(codes==0).mean()*100:5.1f}%  "
                  f"Partial={(codes==1).mean()*100:5.1f}%  "
                  f"Correct={(codes==2).mean()*100:5.1f}%")

    wide = None
    for cfg in CONFIG_ORDER:
        col = frames[cfg][["QueryNumber", metric]].rename(columns={metric: cfg})
        wide = col if wide is None else wide.merge(col, on="QueryNumber")
    wide = wide.dropna()
    print(f"\nComplete-case queries for paired analysis: n = {len(wide)}")
    if not do_tests:
        print("(Inferential tests skipped: RUN_TESTS = False)")
        return

    chi2, p = friedmanchisquare(*[wide[c].values for c in CONFIG_ORDER])
    W = kendalls_w(wide[CONFIG_ORDER].values)
    print(f"Friedman: chi^2({len(CONFIG_ORDER)-1}) = {chi2:.3f}, p = {p:.4g}, "
          f"Kendall's W = {W:.3f}")

    rows = []
    for a, b in combinations(CONFIG_ORDER, 2):
        xa, xb = wide[a].values, wide[b].values
        try:
            _, pw = wilcoxon(xa, xb)
        except ValueError:
            pw = 1.0
        rows.append([f"{a} vs {b}", float(np.mean(xb - xa)), pw, rank_biserial(xa, xb)])
    padj = holm([r[2] for r in rows])
    print("Post-hoc pairwise Wilcoxon (Holm-corrected):")
    print(f"  {'pair':26s} {'mean_diff':>10s} {'p_raw':>8s} {'p_holm':>8s} {'r_rb':>7s} sig")
    for (pair, md_, p0, e), p1 in zip(rows, padj):
        print(f"  {pair:26s} {md_:10.4f} {p0:8.4f} {p1:8.4f} {e:7.3f} "
              f"{'*' if p1 < 0.05 else ''}")

def rq2_agreement(metric, label):
    print("=" * 66)
    print(f"RQ2 -- AGREEMENT: {label} vs human judgment (per response)")
    print("=" * 66)
    print(f"  {'config':10s} {'n':>5s} {'Spearman rho':>13s} {'Kendall tau':>12s}")
    allx, ally = [], []
    for cfg in CONFIG_ORDER:
        d = frames[cfg].dropna(subset=[metric, "human"])
        rho, _ = spearmanr(d[metric], d["human"]); tau, _ = kendalltau(d[metric], d["human"])
        print(f"  {cfg:10s} {len(d):5d} {rho:13.3f} {tau:12.3f}")
        allx.append(d[metric].values); ally.append(d["human"].values)
    X, Y = np.concatenate(allx), np.concatenate(ally)
    rho, _ = spearmanr(X, Y); tau, _ = kendalltau(X, Y)
    print(f"  {'ALL':10s} {len(X):5d} {rho:13.3f} {tau:12.3f}")

## RQ1 — Relevance (BERTScore)

In [ ]:
rq1_analysis("bert", "Relevance (BERTScore)", do_tests=RUN_TESTS)

## RQ1 — Relevance (ROUGE-L)

In [ ]:
if rouge_ok:
    rq1_analysis("rouge", "Relevance (ROUGE-L)", do_tests=RUN_TESTS)
else:
    print("ROUGE-L not available; skipped.")

## RQ1 — Correctness (human, 0–2)

In [ ]:
rq1_analysis("human", "Correctness (human, 0-2)", do_tests=RUN_TESTS)

## RQ2 — Agreement between automatic metrics and human judgment

In [ ]:
rq2_agreement("bert", "BERTScore")
print()
if rouge_ok:
    rq2_agreement("rouge", "ROUGE-L")